# Range Structures - DSA Patterns

Range data structures efficiently handle range queries and updates.

## Topics Covered
1. Prefix Sum (2D)
2. Segment Tree
3. Binary Indexed Tree (Fenwick Tree)
4. Lazy Propagation
5. Range Query Applications

## 1. Prefix Sum

In [ ]:
class PrefixSum:
    """Prefix Sum for range queries."""
    
    def __init__(self, arr):
        self.n = len(arr)
        self.prefix = [0] * (self.n + 1)
        for i in range(self.n):
            self.prefix[i + 1] = self.prefix[i] + arr[i]
    
    def query(self, left, right):
        """Get sum of arr[left..right]."""
        return self.prefix[right + 1] - self.prefix[left]


# Example
arr = [1, 3, 5, 7, 9, 11]
ps = PrefixSum(arr)
print(f"Array: {arr}")
print(f"Sum [1..3]: {ps.query(1, 3)}")
print(f"Sum [0..5]: {ps.query(0, 5)}")

## 2. Segment Tree

In [ ]:
class SegmentTree:
    """Segment Tree for range sum queries and point updates."""
    
    def __init__(self, arr):
        self.n = len(arr)
        self.tree = [0] * (4 * self.n)
        self.build(arr, 0, 0, self.n - 1)
    
    def build(self, arr, node, start, end):
        if start == end:
            self.tree[node] = arr[start]
        else:
            mid = (start + end) // 2
            self.build(arr, 2 * node + 1, start, mid)
            self.build(arr, 2 * node + 2, mid + 1, end)
            self.tree[node] = self.tree[2 * node + 1] + self.tree[2 * node + 2]
    
    def update(self, node, start, end, idx, val):
        """Update value at index idx."""
        if start == end:
            self.tree[node] = val
        else:
            mid = (start + end) // 2
            if idx <= mid:
                self.update(2 * node + 1, start, mid, idx, val)
            else:
                self.update(2 * node + 2, mid + 1, end, idx, val)
            self.tree[node] = self.tree[2 * node + 1] + self.tree[2 * node + 2]
    
    def query(self, node, start, end, left, right):
        """Get sum of range [left, right]."""
        if right < start or end < left:
            return 0
        if left <= start and end <= right:
            return self.tree[node]
        
        mid = (start + end) // 2
        left_sum = self.query(2 * node + 1, start, mid, left, right)
        right_sum = self.query(2 * node + 2, mid + 1, end, left, right)
        return left_sum + right_sum


# Example
arr = [1, 3, 5, 7, 9, 11]
st = SegmentTree(arr)
print(f"Array: {arr}")
print(f"Sum [1..3]: {st.query(0, 0, 5, 1, 3)}")
st.update(0, 0, 5, 2, 10)
print(f"After update index 2 to 10:")
print(f"Sum [1..3]: {st.query(0, 0, 5, 1, 3)}")

## 3. Binary Indexed Tree (Fenwick Tree)

In [ ]:
class BIT:
    """Binary Indexed Tree (Fenwick Tree)."""
    
    def __init__(self, arr):
        self.n = len(arr)
        self.tree = [0] * (self.n + 1)
        for i, val in enumerate(arr):
            self.update(i, val)
    
    def update(self, idx, delta):
        """Add delta to element at index idx."""
        idx += 1
        while idx <= self.n:
            self.tree[idx] += delta
            idx += idx & (-idx)
    
    def query(self, idx):
        """Get sum of arr[0..idx]."""
        result = 0
        idx += 1
        while idx > 0:
            result += self.tree[idx]
            idx -= idx & (-idx)
        return result
    
    def range_query(self, left, right):
        """Get sum of arr[left..right]."""
        if left == 0:
            return self.query(right)
        return self.query(right) - self.query(left - 1)


# Example
arr = [1, 3, 5, 7, 9, 11]
bit = BIT(arr)
print(f"Array: {arr}")
print(f"Sum [1..3]: {bit.range_query(1, 3)}")
print(f"Sum [0..5]: {bit.range_query(0, 5)}")
bit.update(2, 5)  # Add 5 to index 2
print(f"After adding 5 to index 2:")
print(f"Sum [1..3]: {bit.range_query(1, 3)}")

## 4. Lazy Propagation (Range Update)

In [ ]:
class LazySegmentTree:
    """Segment Tree with lazy propagation for range updates."""
    
    def __init__(self, arr):
        self.n = len(arr)
        self.tree = [0] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self.build(arr, 0, 0, self.n - 1)
    
    def build(self, arr, node, start, end):
        if start == end:
            self.tree[node] = arr[start]
        else:
            mid = (start + end) // 2
            self.build(arr, 2 * node + 1, start, mid)
            self.build(arr, 2 * node + 2, mid + 1, end)
            self.tree[node] = self.tree[2 * node + 1] + self.tree[2 * node + 2]
    
    def push(self, node, start, end):
        if self.lazy[node] != 0:
            self.tree[node] += self.lazy[node] * (end - start + 1)
            if start != end:
                self.lazy[2 * node + 1] += self.lazy[node]
                self.lazy[2 * node + 2] += self.lazy[node]
            self.lazy[node] = 0
    
    def range_update(self, node, start, end, left, right, val):
        """Add val to all elements in range [left, right]."""
        self.push(node, start, end)
        if right < start or end < left:
            return
        if left <= start and end <= right:
            self.lazy[node] += val
            self.push(node, start, end)
            return
        
        mid = (start + end) // 2
        self.range_update(2 * node + 1, start, mid, left, right, val)
        self.range_update(2 * node + 2, mid + 1, end, left, right, val)
        self.tree[node] = self.tree[2 * node + 1] + self.tree[2 * node + 2]
    
    def range_query(self, node, start, end, left, right):
        """Get sum of range [left, right]."""
        self.push(node, start, end)
        if right < start or end < left:
            return 0
        if left <= start and end <= right:
            return self.tree[node]
        
        mid = (start + end) // 2
        left_sum = self.range_query(2 * node + 1, start, mid, left, right)
        right_sum = self.range_query(2 * node + 2, mid + 1, end, left, right)
        return left_sum + right_sum


# Example
arr = [1, 3, 5, 7, 9]
lst = LazySegmentTree(arr)
print(f"Array: {arr}")
print(f"Sum [1..3]: {lst.range_query(0, 0, 4, 1, 3)}")
lst.range_update(0, 0, 4, 1, 3, 2)  # Add 2 to range [1..3]
print(f"After adding 2 to range [1..3]:")
print(f"Sum [1..3]: {lst.range_query(0, 0, 4, 1, 3)}")

## 5. Range Query Applications

In [ ]:
def range_sum_queries(arr, queries):
    """Answer multiple range sum queries."""
    ps = PrefixSum(arr)
    results = []
    for left, right in queries:
        results.append(ps.query(left, right))
    return results


def count_inversions_merge_sort(arr):
    """Count inversions using merge sort.
    Time: O(n log n)"""
    def merge_sort(arr, temp, left, right):
        if left >= right:
            return 0
        
        mid = (left + right) // 2
        inversions = 0
        inversions += merge_sort(arr, temp, left, mid)
        inversions += merge_sort(arr, temp, mid + 1, right)
        inversions += merge(arr, temp, left, mid, right)
        return inversions
    
    def merge(arr, temp, left, mid, right):
        i, j, k = left, mid + 1, left
        inversions = 0
        
        while i <= mid and j <= right:
            if arr[i] <= arr[j]:
                temp[k] = arr[i]
                i += 1
            else:
                temp[k] = arr[j]
                j += 1
                inversions += mid - i + 1
            k += 1
        
        while i <= mid:
            temp[k] = arr[i]
            i += 1
            k += 1
        while j <= right:
            temp[k] = arr[j]
            j += 1
            k += 1
        
        for i in range(left, right + 1):
            arr[i] = temp[i]
        return inversions
    
    temp = arr.copy()
    return merge_sort(arr, temp, 0, len(arr) - 1)


# Examples
arr = [1, 3, 5, 7, 9, 11]
queries = [(0, 2), (1, 4), (2, 5)]
print(f"Array: {arr}")
print(f"Range sums: {range_sum_queries(arr, queries)}")

arr = [5, 3, 2, 4, 1]
print(f"\nInversions in {arr}: {count_inversions_merge_sort(arr.copy())}")